<!-- # VMamba: Visual State Space Model -->
# VMamba：视觉状态空间模型

- [论文](https://arxiv.org/abs/2401.10166)
- [代码](https://github.com/MzeroMiko/VMamba?tab=readme-ov-file#main-results)
- [代码复现](https://github.com/LitTTian/VMamba)

## 摘要
设计计算效率高的网络架构始终是计算机视觉领域的核心需求。本文将状态空间语言模型 Mamba 引入视觉领域，提出了具有线性时间复杂度的视觉骨干网络 VMamba。其核心架构由视觉状态空间（VSS）块和 二维选择性扫描（SS2D） 模块堆叠而成。通过四条扫描路径，SS2D 成功弥合了一维选择性扫描的序列性与二维视觉数据非序列性之间的鸿沟，从而能够从多视角、多维度采集上下文信息。在此基础上，我们开发了 VMamba 系列模型，并通过一系列架构与工程优化提升了运行速度。大量实验证明，VMamba 在多项视觉感知任务中表现优异，尤其在输入尺度扩展效率上明显优于现有基准模型。

## 1 引言

视觉表示学习是计算机视觉的基础研究领域。在深度学习时代，卷积神经网络（CNNs）和 Vision Transformers（ViTs）是两大主流骨干网络。相比 CNN，ViT 凭借自注意力机制在大规模数据上展现出更强的学习能力，但自注意力的计算复杂度随 Token 数量呈平方级增长，限制了其在高分辨率场景下的应用。

为了解决这一挑战，研究者们曾尝试改进注意力效率，但往往以缩小感受野或牺牲性能为代价。这促使我们开发一种既能保持全局感受野和动态权重的优势，又能降低计算成本的新架构。

近期，NLP 领域的 Mamba（一种新型状态空间模型 SSM）凭借线性复杂度的长序列建模能力脱颖而出。受此启发，我们提出了 VMamba。由于 Mamba 原生的选择性扫描算法是为一维序列设计的，难以直接处理非序列性的二维视觉数据，我们专门设计了 二维选择性扫描（SS2D） 机制。SS2D 通过四向扫描遍历空间域，将计算复杂度从平方级降低至线性。

基于视觉状态空间（VSS）块，我们构建了 VMamba 系列模型（Tiny/Small/Base）。实验表明，VMamba 在 ImageNet-1K 分类、COCO 检测和 ADE20K 分割等任务中均优于 Swin Transformer 和 ConvNeXt。例如，VMamba-Base 在 ImageNet 上达到 83.9% 的准确率，且吞吐量比 Swin 高出 40% 以上。

主要贡献如下：
1. 提出 VMamba 架构： 我们提出了一种基于状态空间模型（SSM）的视觉骨干网络 VMamba。该架构不仅实现了线性时间复杂度，还通过一系列架构改进和工程优化，显著提升了模型的推理速度。
2. 引入二维选择性扫描机制（SS2D）： 为解决 SSM 难以处理非序列化图像数据的问题，我们设计了 SS2D 模块。它成功桥接了一维数组扫描与二维平面遍历，使选择性状态空间模型能够高效地处理视觉任务。
3. 卓越的性能与扩展性： VMamba 在图像分类、目标检测和语义分割等多种视觉任务中均表现优异。此外，该模型对输入序列长度具有极强的适应性，计算量（FLOPs）仅随输入规模线性增长，展现了顶尖的输入扩展能力。

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/ss2d_linear_complexity.png" />
    <span style="font-size: 12px; color: black;">图1</strong>：通过（a）自注意力机制和（b）所提出的二维选择性扫描（SS2D）建立图像块间相关性的对比。红色框表示查询图像块，其不透明度代表信息丢失程度。</span>
</div>

## 2 相关工作

<!-- __卷积神经网络（CNNs）__
自AlexNet提出以来，研究人员围绕**提升模型建模能力**与**计算效率**，在各类视觉任务中对CNN展开了大量优化。深度可分离卷积、可变形卷积等复杂算子的引入，进一步增强了CNN的灵活性与有效性。近年来，受Transformer成功的启发，现代CNN通过融合**长距离依赖建模**与**动态权重机制**，取得了极具竞争力的性能。

__视觉Transformer（ViTs）__
ViT作为开创性工作，验证了纯Transformer架构在视觉任务中的有效性，并强调**大规模预训练**对图像分类性能的关键作用。为降低ViT对大规模数据集的依赖，DeiT提出师生蒸馏策略，将CNN的知识迁移至ViT，凸显了**归纳偏置**在视觉感知中的重要性。后续研究在此基础上，进一步提出了**分层结构ViT**。

ViT的另一研究方向聚焦于**自注意力机制的计算效率优化**。线性注意力通过核特征图的线性点积重构自注意力，利用矩阵乘积的结合律将计算复杂度从二次级降至线性级。GLA设计了硬件友好的线性注意力变体，实现了内存占用与并行性的平衡。RWKV结合线性注意力，兼顾了Transformer的并行训练与循环神经网络（RNN）的高效推理。RetNet引入门控机制构建并行计算路径，为循环结构提供了替代方案。RMT则将时间衰减机制拓展至空间域，进一步推动了线性注意力在视觉表征学习中的应用。

__状态空间模型（SSMs）__
尽管ViT在视觉任务中应用广泛，但其自注意力机制的**二次级复杂度**在处理长序列输入（如高分辨率图像）时面临瓶颈。为提升模型的扩展效率，状态空间模型（SSMs）作为Transformer的有力替代方案，受到了广泛关注。Gu等人基于HiPPO初始化，验证了SSM在长距离依赖建模中的潜力。为提升实用性，S4将参数矩阵归一化为对角结构。此后，一系列结构化SSM模型相继出现，通过**复对角结构**、**多输入多输出支持**、**对角加低秩分解**、**选择机制**等架构改进，进一步提升了模型性能。这些成果已被整合到更大规模的表征模型中，充分体现了结构化SSM的通用性与可扩展性。

值得注意的是，现有SSM模型主要针对文本、语音等**长序列一维数据**设计，针对**二维结构视觉数据**的应用研究仍相对匮乏。 -->
1. **卷积神经网络 (CNNs)**：自 AlexNet 以来，研究重点在于增强 CNN 的建模能力与计算效率。
- 演进： 从 VGG、ResNet 到 DenseNet，引入了深度可分离卷积（Depth-wise Conv）和可变形卷积（Deformable Conv）以提升灵活性。
- 现状： 受 Transformer 启发，现代 CNN（如 ConvNeXt）开始引入长程依赖和动态权重设计。
2. **视觉 Transformer (ViTs)**：ViT 将 Transformer 架构引入视觉领域，展示了大模型在大规模预训练下的潜力。
- 优化途径：DeiT 引入蒸馏策略以降低对大数据的依赖；Swin 等模型提出了层次化架构。
- 计算效率：为解决自注意力的平方复杂度问题，研究者提出了**线性注意力（Linear Attention）** 机制（如 RWKV、RetNet、RMT），尝试在保持并行训练的同时实现高效推理。
3. **状态空间模型 (SSMs)**：针对高分辨率图像带来的计算压力，SSM 成为 Transformer 的有力竞争方案。
- 发展脉络： 从解决长程依赖的 HiPPO，到优化参数结构的 S4（将矩阵对角化），再到 S5、H3 等改进模型。
- 核心突破： Mamba 引入了选择性机制（Selection Mechanism），进一步提升了 SSM 在处理长序列时的表现。
- 视觉应用： 尽管 SSM 在文本和语音领域表现卓越，但在处理具有二维结构的视觉数据方面，目前的研究仍相对有限。

## 3 预备知识

__1. SSM 的公式化 (Formulation)__

SSM源于卡尔曼滤波，是一种**线性非时变 (LTI - linear time-invariant)** 系统。它通过隐藏状态$\mathbf{h}(t) \in \mathbb{R}^{N}$将输入信号$u(t) \in \mathbb{R}$映射为输出响应$y(t) \in \mathbb{R}$。其连续时间的线性常微分方程 (ODE) 表示为：
$$\begin{equation}\tag{1}
\begin{split}
  \mathbf{h'}(t) &= \mathbf{A} \mathbf{h}(t) + \mathbf{B} u(t),  \\
  y(t)  &= \mathbf{C} \mathbf{h}(t) + D u(t),
\end{split}
\end{equation}$$
其中 $\mathbf{A} \in \mathbb{R}^{N\times N}$、$\mathbf{B}\in \mathbb{R}^{N\times 1}$、$\mathbf{C}\in \mathbb{R}^{1\times N}$ 和 $D \in \mathbb{R}^{1}$ 为模型的权重参数。

__2. 离散化 (Discretization)__

<!-- * **核心逻辑：** 通过引入步长参数 （步长），将连续的 ODE 转化为离散的序列。
* **方法：** 通常采用**零阶保持 (ZOH)** 手段，将连续状态方程转换为离散形式 ，使其能够在计算机上高效处理。 -->

为集成到深度模型中，**连续时间SSM必须先进行离散化**。

对于时间区间 $[t_a, t_b]$，隐藏状态 $\mathbf{h}(t)$ 在 $t = t_b$ 时刻的解析解为：
$$\begin{equation}\tag{2}
    \mathbf{h}(t_b) = e^{\mathbf{A}(t_b-t_a)} \mathbf{h}(t_a) + e^{\mathbf{A}(t_b - t_a)} \int_{t_a}^{t_b} \mathbf{B}(\tau) u(\tau) e^{-\mathbf{A}(\tau-t_a)} \,d\tau.
\end{equation}$$

通过时间尺度参数 $\boldsymbol{\Delta}$ 采样（即 $d\tau |_{t_i}^{t_{i+1}} =\Delta_i$ ），可得到离散形式：
$$\begin{equation}\tag{3}
    \mathbf{h}_b = e^{\mathbf{A}(\Delta_a+...+\Delta_{b-1})} \left( \mathbf{h}_a + \sum^{b-1}_{i=a} \mathbf{B}_i u_i e^{-\mathbf{A}(\Delta_a+...+\Delta_i)} \Delta_i \right),
\end{equation}$$
其中 $[a,b]$ 为对应的离散步长区间。需注意，该形式**近似于零阶保持（ZOH）方法**的计算结果，后者在SSM相关研究中被广泛采用（详细证明见附录[A]()）。

__选择性扫描机制 (Selective Scan)__

针对线性时不变SSM（[公式1]()）**难以捕捉上下文信息**的局限，Gu等人提出了一种新的SSM参数化方法，引入了**输入依赖的选择机制**（简称S6）。然而，选择性SSM的**时变权重参数**给隐藏状态的高效计算带来了挑战——卷积操作无法兼容动态权重，因此不再适用。尽管如此，由于[公式3]()中 $h_b$ 的递推关系可被推导，输出响应 $y_b$ 仍可通过**结合扫描算法**实现高效计算，复杂度为线性（详细推导见附录[B]()）。

## 4 VMamba: 视觉状态空间模型

### 4.1 网络架构

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/architecture.png" />
    <span style="font-size: 12px; color: black;">图3</strong>：左图：（a）VMamba的整体架构示意图，以及（b）-（d）Mamba和VSS块的结构。<br>右图：VMamba变体与基准方法在分类精度和计算效率方面的比较。</span>
</div>

我们设计了三种尺度的VMamba模型：Tiny、Small和Base（分别记为VMamba-T、VMamba-S、VMamba-B）。VMamba-T的整体架构如[图3]() (a)所示，详细配置见附录[E]()。

输入图像 $\mathbf{I}\in \mathbb{R}^{H\times W\times 3}$ 首先通过主干模块（stem module）分割为图像块，得到空间维度为 $H/4\times W/4$ 的二维特征图。无需额外引入位置编码，模型通过多个网络阶段生成空间分辨率为 $H/8\times W/8$、$H/16\times W/16$ 和 $H/32\times W/32$ 的层级表征。具体而言，每个阶段包含一个下采样层（第一阶段除外），后续堆叠多个视觉状态空间（VSS）块。

VSS块是Mamba块在视觉任务中的对应结构（[图3]() (b)），用于表征学习。初始VSS块（[图3]() (c) 中的“基础VSS块”）通过替换Mamba的核心模块S6构建——S6模块是Mamba实现全局感受野、动态权重（即选择性）和线性复杂度的关键。我们将其替换为新提出的二维选择性扫描（SS2D）模块，细节将在下一小节介绍。

为进一步提升计算效率，我们移除了整个乘法分支（[图3]() (c) 中红色框标注部分），因为SS2D的选择性已实现门控机制的功能。最终，改进后的VSS块（[图3]() (d)）仅包含单个网络分支及两个残差模块，结构类似基础Transformer块。本文所有实验结果均基于该架构的VSS块构建的VMamba模型获得。


### 4.2 用于视觉数据的二维选择性扫描（SS2D）

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/cross_scan.png" />
    <span style="font-size: 12px; color: black;">图2</strong>：二维选择性扫描（SS2D）示意图。输入Patches沿着四条不同的扫描路径（交叉扫描）进行遍历，每个序列由独立的S6块进行处理。然后将结果合并，构建二维特征图作为最终输出（交叉合并）。</span>
</div>

S6 的序列扫描特性适配自然语言处理（NLP）中的时序数据，但难以应对视觉数据 —— 视觉数据本质是非序列的，包含空间信息（如局部纹理、全局结构）。为解决该问题，S4ND通过卷积操作重构SSM，利用外积将核从一维直接扩展到二维。然而，这种修改导致权重失去输入依赖性，捕捉上下文信息的能力受限。因此，我们沿用选择性扫描方法处理输入，提出二维选择性扫描（SS2D）模块，在保留S6优势的前提下使其适配视觉数据。
如[图2]()所示，SS2D 的数据流包含三个步骤：交叉扫描、S6块选择性扫描、交叉融合。具体而言，SS2D 首先将输入图像块沿四条不同遍历路径展开为序列（即交叉扫描）；随后，每条序列由独立的 S6 块并行处理；最终，将处理后的序列重塑并合并，生成输出特征图（即交叉融合）。通过互补的一维遍历路径，图像中每个像素可整合不同方向上所有其他像素的信息，从而在二维空间中建立全局感受野。

### 4.3 加速VMamba

<div style="background-color:white; padding:10px; border-radius:5px; margin: auto; text-align: center; overflow: hidden; width: 80%;">
    <div style="width: 100%; overflow: hidden; border:1px solid #ccc;">
        <img src="./assets/architecture.png" style="margin-left: -60%;" />
    </div>
    <span style="font-size: 12px; color: black; margin-top: 10px; display: block;">
        <strong>图3（右）</strong>：VMamba 变体与基准方法在分类精度和计算效率方面的比较。
    </span>
</div>

如[图3]() (e)所示，采用基础VSS块的VMamba-T模型（简称**基础版VMamba**）的推理吞吐量为426张/秒，参数量2290万，计算量56亿次浮点运算（FLOPs）。尽管该模型在小尺度模型中实现了82.2%的SOTA分类精度（较Swin-T提升0.9%），但**低吞吐量与高内存开销**为其实际部署带来了巨大挑战。

本节将详细介绍我们提升模型推理速度的优化策略，核心聚焦于**实现细节优化**与**架构设计改进**两大方向。所有模型均基于ImageNet-1K图像分类任务进行评估，各渐进式优化步骤的性能增益总结如下（括号内数值分别表示ImageNet-1K的Top-1精度提升率和推理吞吐量提升量，单位：%、张/秒）。相关细节拓展见[附录E]()。

1.  **步骤(a)**：(+0.0%, +41)
    基于**Triton框架**重实现交叉扫描（Cross-Scan）与交叉融合（Cross-Merge）模块。
2.  **步骤(b)**：(+0.0%, -3)
    调整选择性扫描的CUDA实现，使其支持**float16输入与float32输出**。尽管测试阶段吞吐量略有波动，但训练效率显著提升（训练吞吐量从165张/秒提升至184张/秒）。
3.  **步骤(c)**：(+0.0%, +174)
    用**线性变换**（即`torch.nn.functional.linear`）替换选择性扫描中速度较慢的`einsum`操作；同时采用 **(B, C, H, W)张量格式** ，消除不必要的数据维度变换。
4.  **步骤(d)**：(-0.6%, +175)
    引入计算高效的**MLP模块**，移除深度可分离卷积（DWConv[2]）层；将网络层配置从`[2,2,9,2]`调整为`[2,2,2,2]`，以降低计算量。
5.  **步骤(e)**：(+0.6%, +366)
    将特征扩展因子 **`ssm-ratio`** 从2.0降至1.0（简称步骤d.1）；将网络层配置增加至`[2,2,5,2]`（简称步骤d.2）；移除[图3]() (c)中所示的整个乘法分支。
6.  **步骤(f)**：(+0.3%, +161)
    重新引入**DWConv层**（简称步骤e.1）；将SSM状态维度 **`d_state`** 从16.0降至1.0（简称步骤e.2）；同时将`ssm-ratio`恢复至2.0。
7.  **步骤(g)**：(+0.1%, +346)
    将`ssm-ratio`重新降至1.0；将网络层配置从`[2,2,5,2]`调整为`[2,2,8,2]`。

## 5 实验

本节通过一系列实验，在**多类视觉任务**中评估VMamba的性能并与主流基准模型对比；同时，通过与其他方法的对比，验证所提**二维特征图遍历方法**的有效性。此外，我们还通过可视化**有效感受野（ERF）**与激活图、分析模型在**长输入序列下的可扩展性**，对VMamba的特性展开深入分析。

实验超参数与配置**主要沿用Swin模型**的设置。详细实验配置见[附录 E]()与[附录 F]()，额外消融实验结果见附录[附录 H]()。所有实验均在配备**8张NVIDIA Tesla A100 GPU**的服务器上完成。

### 5.1 图像分类

我们在[ImageNet-1K数据集]()上评估VMamba的图像分类性能，与基准模型的对比结果汇总于[表1]()。

在**计算量（FLOPs）相当**的前提下，VMamba-T的Top-1分类精度达82.6%，较DeiT-S[2]提升2.8个百分点，较Swin-T[3]提升1.3个百分点。值得注意的是，VMamba在Small和Base尺度下仍保持性能优势——例如VMamba-B的Top-1精度为83.9%，分别超越DeiT-B[2]和Swin-B[3] 2.1和0.4个百分点。

**计算效率方面**，VMamba-T的推理吞吐量达1686张/秒，性能优于或媲美当前SOTA方法。这一优势在更大尺度模型中持续体现：VMamba-S和VMamba-B的吞吐量分别为877张/秒和646张/秒。与基于SSM的模型相比，VMamba-T的吞吐量是S4ND-Conv-T[4]的1.47倍、Vim-S[5]的1.08倍，同时分类精度仍分别领先0.4和2.1个百分点。

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; display: grid; 
    grid-template-columns: 1fr 1fr; /* 两列均等 */ 
    grid-template-rows: auto auto; /* 第一行图片，第二行标题 */ 
    gap: 10px 20px; /* 行间距 10px，列间距 20px */ 
align-items: start; justify-items: center;">
    <div style="font-size: 12px; color: black; width: 100%; text-align: center;grid-column: span 2;">
        <strong>表1</strong>：在ImageNet-1K上的性能比较。吞吐量值是使用A100 GPU和AMD EPYC 7542 CPU，采用发布的工具包，并遵循中提出的协议测量的。所有图像的尺寸均为224x224。
    </div>
    <img src="./assets/imagenet_result1.png" style="max-width: 100%; height: auto;" />
    <img src="./assets/imagenet_result2.png" style="max-width: 100%; height: auto;" />
</div>

### 5.2 下游任务

在本节中，我们评估了VMamba在下游任务上的性能，包括在MSCOCO2017上的目标检测和实例分割，以及在ADE20K上的语义分割。训练框架基于MMDetection和MMSegmenation库，遵循的方法，分别使用Mask R-CNN和UperNet作为检测网络和分割网络。

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; display: grid; 
    grid-template-columns: 1fr 1fr; /* 两列均等 */ 
    grid-template-rows: auto auto; /* 第一行图片，第二行标题 */ 
    gap: 10px 20px; /* 行间距 10px，列间距 20px */ 
align-items: start; justify-items: center;">
    <div style="font-size: 12px; color: black; width: 100%; text-align: center;grid-column: span 2;">
        <strong>表2</strong>：左图：MSCOCO上目标检测和实例分割的结果。APᵇ和APᵐ分别表示边界框AP和掩码AP。FLOPs的计算输入尺寸为1280x800。符号“1X”表示模型微调了12个 epoch，而“3X”表示多尺度训练了36个epoch。右图：ADE20K上语义分割的结果。FLOPs的计算输入尺寸为512x2048。“SS”和“MS”分别表示单尺度测试和多尺度测试。
    </div>
    <img src="./assets/mscoco_result1.png" style="max-width: 100%; height: auto;" />
    <img src="./assets/mscoco_result2.png" style="max-width: 100%; height: auto;" />
</div>

__目标检测与实例分割__  
MSCOCO数据集上的实验结果如[表2]()所示。**VMamba在不同训练策略下，均在框平均精度（APᵇ）与掩码平均精度（APᵐ）上表现出显著优势**。

在12轮微调策略下，VMamba-T/S/B的目标检测平均精度均值（mAP）分别为47.3%/48.7%/49.2%，较Swin-T/S/B分别提升4.6%/3.9%/2.3%，较ConvNeXt-T/S/B分别提升3.1%/3.3%/2.2%。实例分割任务中，VMamba-T/S/B的mAP较Swin-T/S/B分别领先3.4%/2.8%/1.8%，较ConvNeXt-T/S/B分别领先2.6%/1.9%/1.4%。

即使在**36轮多尺度训练微调策略**下，VMamba的性能优势依然保持，充分证明其在密集预测类下游任务中具备强大潜力。

__语义分割__  
与前文实验结论一致，**在参数量相当的前提下，VMamba在ADE20K语义分割任务中仍表现最优**。如表[表2]()所示，在单尺度（SS）输入设置下，VMamba-T的平均交并比（mIoU）较Swin-T提升3.4%，较ConvNeXt-T提升1.9%；该优势在多尺度（MS）输入设置下同样成立。

对于Small和Base尺度模型，单尺度设置下VMamba-S/B的mIoU较NAT-S/B[1]分别提升2.6%/2.5%；多尺度设置下，优势分别为1.7%/1.9%。

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/scale_resolution_acc.png" />
    <span style="font-size: 12px; color: black;">图4</strong>：VMamba对（a）下游任务和（b）分辨率逐渐增加的输入图像的适应性说明。Swin-T∗表示使用缩放窗口大小测试的Swin-T。</span>
</div>

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; text-align: center;">
    <img src="./assets/scale_resolution_cost.png" />
    <span style="font-size: 12px; color: black;">图5</strong>：VMamba的资源消耗随分辨率逐步增加的说明。Swin-T∗表示使用缩放窗口大小测试的Swin-T。</span>
</div>

__讨论__  
本节实验结果验证了**VMamba对目标检测、实例分割与语义分割任务的良好适应性**。[图4]() (a)对比了VMamba与Swin、ConvNeXt的性能，结果显示：在ImageNet-1K分类精度相当的前提下，VMamba在下游任务中优势显著。

这一结论与[图4]() (b)的结果一致——VMamba在不同输入图像尺寸下表现出**最优的性能稳定性**（即性能下降幅度最小）。在输入分辨率为768×768时，VMamba无需微调即可实现74.7%的Top-1分类精度，线性微调后精度可达79.2%。

值得注意的是，VMamba在对输入分辨率变化具备更强鲁棒性的同时，**计算量（FLOPs）与内存消耗仍保持线性增长**（见[图5]() (a)、(c)），且吞吐量始终维持在较高水平（见[图5]() (b)）。这使得VMamba在适配大空间分辨率输入的下游任务时，相比基于ViT的方法兼具更高的有效性与效率。

该特性与Mamba模型在**高效长序列建模**方面的先天优势高度契合。